# Amazon Fashion Collaborative Filtering Baseline (Scalable)

This notebook builds a production-friendly collaborative filtering baseline with sparse latent factors and ranking metrics (**HR@10**, **NDCG@10**, **MRR@10**).


## Model Choice

Primary backend: **`implicit` ALS** (optimized for sparse interaction data).

Fallback backend: **sparse `TruncatedSVD`** from scikit-learn when `implicit` is not installed in the runtime.


In [13]:
# Optional if your environment doesn't have implicit installed:
%pip install implicit



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix
from sklearn.decomposition import TruncatedSVD
from tqdm.auto import tqdm

IMPLICIT_AVAILABLE = True
try:
    from implicit.als import AlternatingLeastSquares
except ImportError:
    IMPLICIT_AVAILABLE = False
    AlternatingLeastSquares = None

print(f"implicit available: {IMPLICIT_AVAILABLE}")


implicit available: True


In [16]:
@dataclass
class PipelineConfig:
    train_path: str = "mini_train.csv"
    test_path: str = "mini_test.csv"
    file_format: str = "csv"  # change to 'parquet' for production parquet files

    user_col: str = "user_id"
    item_col: str = "parent_asin"
    rating_col: str = "rating"

    rating_max: float = 5.0
    positive_threshold: float = 4.0

    # backend: 'auto' (prefer ALS), 'als', or 'svd'
    backend: str = "auto"

    # factorization params
    factors: int = 64
    regularization: float = 0.05
    iterations: int = 15
    alpha: float = 40.0
    random_state: int = 42

    # ranking eval
    k: int = 10
    batch_size: int = 128
    max_eval_users: Optional[int] = None  # set a number for quicker debug loops


cfg = PipelineConfig()
cfg


PipelineConfig(train_path='mini_train.csv', test_path='mini_test.csv', file_format='csv', user_col='user_id', item_col='parent_asin', rating_col='rating', rating_max=5.0, positive_threshold=4.0, backend='auto', factors=64, regularization=0.05, iterations=15, alpha=40.0, random_state=42, k=10, batch_size=128, max_eval_users=None)

In [17]:
def resolve_data_path(path: str) -> Path:
    raw = Path(path).expanduser()

    if raw.is_absolute() and raw.exists():
        return raw

    cwd = Path.cwd()
    candidates: List[Path] = []

    if raw.is_absolute():
        candidates.append(raw)
    else:
        # direct relative lookups
        candidates.append(cwd / raw)
        candidates.append(cwd / raw.name)

        # common project layout lookup: research/<file>
        candidates.append(cwd / "research" / raw)
        candidates.append(cwd / "research" / raw.name)

        # walk up parent dirs and retry common layouts
        for parent in cwd.parents:
            candidates.append(parent / raw)
            candidates.append(parent / raw.name)
            candidates.append(parent / "research" / raw)
            candidates.append(parent / "research" / raw.name)

    seen = set()
    deduped: List[Path] = []
    for c in candidates:
        key = str(c)
        if key in seen:
            continue
        seen.add(key)
        deduped.append(c)

    for c in deduped:
        if c.exists():
            return c

    tried = "\n".join(f"- {c}" for c in deduped[:12])
    raise FileNotFoundError(
        f"Could not find data file: {path!r}.\n"
        f"Current working directory: {cwd}\n"
        f"Checked these locations:\n{tried}"
    )


def read_interactions(path: str, file_format: str = "csv") -> pd.DataFrame:
    resolved = resolve_data_path(path)
    if file_format == "csv":
        return pd.read_csv(resolved)
    if file_format == "parquet":
        return pd.read_parquet(resolved)
    raise ValueError(f"Unsupported file_format={file_format!r}. Use 'csv' or 'parquet'.")


def prepare_interactions(df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    required = [cfg.user_col, cfg.item_col, cfg.rating_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    out = df[required].copy()
    out[cfg.rating_col] = pd.to_numeric(out[cfg.rating_col], errors="coerce")
    out = out.dropna(subset=required)

    # Deduplicate user-item pairs to reduce matrix nnz.
    out = out.groupby([cfg.user_col, cfg.item_col], as_index=False)[cfg.rating_col].max()

    out[cfg.user_col] = out[cfg.user_col].astype(str)
    out[cfg.item_col] = out[cfg.item_col].astype(str)
    out[cfg.rating_col] = out[cfg.rating_col].astype(np.float32)
    return out


def build_mappings(train_df: pd.DataFrame, cfg: PipelineConfig):
    idx_to_user = pd.Index(train_df[cfg.user_col].unique(), dtype="object")
    idx_to_item = pd.Index(train_df[cfg.item_col].unique(), dtype="object")

    user_to_idx = {u: i for i, u in enumerate(idx_to_user)}
    item_to_idx = {it: i for i, it in enumerate(idx_to_item)}
    return user_to_idx, item_to_idx, idx_to_user, idx_to_item


def encode_interactions(
    df: pd.DataFrame,
    cfg: PipelineConfig,
    user_to_idx: Dict[str, int],
    item_to_idx: Dict[str, int],
    drop_unknown: bool = True,
) -> pd.DataFrame:
    out = df.copy()
    out["user_idx"] = out[cfg.user_col].map(user_to_idx)
    out["item_idx"] = out[cfg.item_col].map(item_to_idx)

    if drop_unknown:
        out = out.dropna(subset=["user_idx", "item_idx"])

    out["user_idx"] = out["user_idx"].astype(np.int32)
    out["item_idx"] = out["item_idx"].astype(np.int32)
    out[cfg.rating_col] = out[cfg.rating_col].astype(np.float32)
    return out


def build_sparse_matrices(
    encoded_train: pd.DataFrame,
    n_users: int,
    n_items: int,
    cfg: PipelineConfig,
) -> Tuple[csr_matrix, csr_matrix]:
    ratings = encoded_train[cfg.rating_col].to_numpy(dtype=np.float32)
    confidence = 1.0 + cfg.alpha * (ratings / cfg.rating_max)

    rows = encoded_train["user_idx"].to_numpy(dtype=np.int32)
    cols = encoded_train["item_idx"].to_numpy(dtype=np.int32)

    user_items = coo_matrix(
        (confidence, (rows, cols)),
        shape=(n_users, n_items),
        dtype=np.float32,
    ).tocsr()

    item_users = user_items.T.tocsr()
    return user_items, item_users


def pick_backend(cfg: PipelineConfig) -> str:
    if cfg.backend not in {"auto", "als", "svd"}:
        raise ValueError("cfg.backend must be one of {'auto', 'als', 'svd'}")

    if cfg.backend == "als":
        if not IMPLICIT_AVAILABLE:
            raise ImportError("cfg.backend='als' but implicit is not installed.")
        return "als"

    if cfg.backend == "svd":
        return "svd"

    return "als" if IMPLICIT_AVAILABLE else "svd"


def train_model(user_items: csr_matrix, cfg: PipelineConfig) -> Dict:
    backend = pick_backend(cfg)

    if backend == "als":
        model = AlternatingLeastSquares(
            factors=cfg.factors,
            regularization=cfg.regularization,
            iterations=cfg.iterations,
            random_state=cfg.random_state,
        )
        model.fit(user_items, show_progress=True)
        return {"backend": "als", "model": model}

    max_rank = min(user_items.shape) - 1
    if max_rank < 2:
        raise ValueError("User-item matrix is too small for SVD fallback.")

    n_components = min(cfg.factors, max_rank)
    svd = TruncatedSVD(n_components=n_components, n_iter=7, random_state=cfg.random_state)
    user_factors = svd.fit_transform(user_items).astype(np.float32)
    item_factors = svd.components_.T.astype(np.float32)

    return {
        "backend": "svd",
        "model": svd,
        "user_factors": user_factors,
        "item_factors": item_factors,
    }


def _ideal_dcg(num_relevant: int, k: int) -> float:
    upto = min(num_relevant, k)
    if upto <= 0:
        return 0.0
    ranks = np.arange(1, upto + 1)
    return float(np.sum(1.0 / np.log2(ranks + 1)))


def _svd_recommend_batch(
    user_factors: np.ndarray,
    item_factors: np.ndarray,
    user_items: csr_matrix,
    batch_users: np.ndarray,
    k: int,
) -> np.ndarray:
    scores = user_factors[batch_users] @ item_factors.T

    for row_id, u in enumerate(batch_users):
        seen = user_items[int(u)].indices
        scores[row_id, seen] = -np.inf

    topk_idx = np.argpartition(scores, -k, axis=1)[:, -k:]
    topk_scores = np.take_along_axis(scores, topk_idx, axis=1)
    order = np.argsort(topk_scores, axis=1)[:, ::-1]
    return np.take_along_axis(topk_idx, order, axis=1)


def evaluate_ranking(
    model_artifacts: Dict,
    user_items: csr_matrix,
    encoded_test: pd.DataFrame,
    cfg: PipelineConfig,
) -> Dict[str, float]:
    positives = encoded_test[encoded_test[cfg.rating_col] >= cfg.positive_threshold]
    user_to_relevant = positives.groupby("user_idx")["item_idx"].agg(set).to_dict()

    eval_users = np.array(sorted(user_to_relevant.keys()), dtype=np.int32)
    if cfg.max_eval_users is not None:
        eval_users = eval_users[: cfg.max_eval_users]

    hr_scores: List[float] = []
    ndcg_scores: List[float] = []
    mrr_scores: List[float] = []

    backend = model_artifacts["backend"]

    for start in tqdm(range(0, len(eval_users), cfg.batch_size), desc="Evaluating"):
        batch_users = eval_users[start : start + cfg.batch_size]

        if backend == "als":
            batch_user_items = user_items[batch_users]
            try:
                rec_items, _ = model_artifacts["model"].recommend(
                    userid=batch_users,
                    user_items=batch_user_items,
                    N=cfg.k,
                    filter_already_liked_items=True,
                )
                rec_items = np.asarray(rec_items)
                if rec_items.ndim == 1:
                    rec_items = rec_items.reshape(1, -1)
            except Exception:
                rec_list = []
                for u in batch_users:
                    ids, _scores = model_artifacts["model"].recommend(
                        userid=int(u),
                        user_items=user_items[int(u)],
                        N=cfg.k,
                        filter_already_liked_items=True,
                    )
                    rec_list.append(np.asarray(ids, dtype=np.int32))
                rec_items = np.vstack(rec_list)
        else:
            rec_items = _svd_recommend_batch(
                user_factors=model_artifacts["user_factors"],
                item_factors=model_artifacts["item_factors"],
                user_items=user_items,
                batch_users=batch_users,
                k=cfg.k,
            )

        for row_id, u in enumerate(batch_users):
            recommended = [int(i) for i in rec_items[row_id].tolist() if int(i) >= 0]
            relevant = user_to_relevant.get(int(u), set())

            hit_ranks = [rank for rank, item_idx in enumerate(recommended, start=1) if item_idx in relevant]

            hr_scores.append(1.0 if hit_ranks else 0.0)
            mrr_scores.append((1.0 / hit_ranks[0]) if hit_ranks else 0.0)

            dcg = 0.0
            for rank, item_idx in enumerate(recommended, start=1):
                if item_idx in relevant:
                    dcg += 1.0 / np.log2(rank + 1)

            idcg = _ideal_dcg(len(relevant), cfg.k)
            ndcg_scores.append((dcg / idcg) if idcg > 0 else 0.0)

    if len(eval_users) == 0:
        return {f"HR@{cfg.k}": 0.0, f"NDCG@{cfg.k}": 0.0, f"MRR@{cfg.k}": 0.0, "eval_users": 0}

    return {
        f"HR@{cfg.k}": float(np.mean(hr_scores)),
        f"NDCG@{cfg.k}": float(np.mean(ndcg_scores)),
        f"MRR@{cfg.k}": float(np.mean(mrr_scores)),
        "eval_users": int(len(eval_users)),
    }


def recommend_for_user(
    model_artifacts: Dict,
    user_id: str,
    user_to_idx: Dict[str, int],
    idx_to_item: pd.Index,
    user_items: csr_matrix,
    k: int = 10,
) -> pd.DataFrame:
    if user_id not in user_to_idx:
        return pd.DataFrame(columns=["parent_asin", "score"])

    u = user_to_idx[user_id]

    if model_artifacts["backend"] == "als":
        item_ids, scores = model_artifacts["model"].recommend(
            userid=int(u),
            user_items=user_items[int(u)],
            N=k,
            filter_already_liked_items=True,
        )
    else:
        user_vec = model_artifacts["user_factors"][int(u)]
        item_factors = model_artifacts["item_factors"]
        scores_all = user_vec @ item_factors.T
        seen = user_items[int(u)].indices
        scores_all[seen] = -np.inf

        topk_idx = np.argpartition(scores_all, -k)[-k:]
        topk_scores = scores_all[topk_idx]
        order = np.argsort(topk_scores)[::-1]
        item_ids = topk_idx[order]
        scores = topk_scores[order]

    return pd.DataFrame(
        {
            "parent_asin": [idx_to_item[i] for i in item_ids],
            "score": scores,
        }
    )


def run_pipeline(cfg: PipelineConfig):
    train_raw = read_interactions(cfg.train_path, cfg.file_format)
    test_raw = read_interactions(cfg.test_path, cfg.file_format)

    train_df = prepare_interactions(train_raw, cfg)
    test_df = prepare_interactions(test_raw, cfg)

    user_to_idx, item_to_idx, idx_to_user, idx_to_item = build_mappings(train_df, cfg)

    train_enc = encode_interactions(train_df, cfg, user_to_idx, item_to_idx, drop_unknown=True)
    test_enc = encode_interactions(test_df, cfg, user_to_idx, item_to_idx, drop_unknown=True)

    n_users = len(idx_to_user)
    n_items = len(idx_to_item)

    user_items, _ = build_sparse_matrices(train_enc, n_users, n_items, cfg)

    model_artifacts = train_model(user_items, cfg)
    metrics = evaluate_ranking(model_artifacts, user_items, test_enc, cfg)

    artifacts = {
        "model_artifacts": model_artifacts,
        "user_items": user_items,
        "train_df": train_df,
        "test_df": test_df,
        "train_enc": train_enc,
        "test_enc": test_enc,
        "user_to_idx": user_to_idx,
        "item_to_idx": item_to_idx,
        "idx_to_user": idx_to_user,
        "idx_to_item": idx_to_item,
        "metrics": metrics,
    }
    return artifacts


In [18]:
# For a quick dry-run, set cfg.max_eval_users to a smaller value (e.g., 2000).
# cfg.max_eval_users = 2000

artifacts = run_pipeline(cfg)
print('backend used:', artifacts['model_artifacts']['backend'])
artifacts["metrics"]


/usr/local/python/3.12.1/lib/python3.12/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
Evaluating: 100%|██████████| 95/95 [00:09<00:00,  9.85it/s]

backend used: als


{'HR@10': 0.018566100290095318,
 'NDCG@10': 0.008848698821904347,
 'MRR@10': 0.007897104967142266,
 'eval_users': 12065}

In [19]:
sample_users = artifacts["idx_to_user"][:3].tolist()
for uid in sample_users:
    print(f"\nUser: {uid}")
    display(
        recommend_for_user(
            model_artifacts=artifacts["model_artifacts"],
            user_id=uid,
            user_to_idx=artifacts["user_to_idx"],
            idx_to_item=artifacts["idx_to_item"],
            user_items=artifacts["user_items"],
            k=cfg.k,
        )
    )



User: AE222KSFP7UXSULU6I34OU2A2DUQ


,parent_asin,score
0,B0084AVW7I,0.303954
1,B003VPA6BO,0.225773
2,B0040JGADS,0.213956
3,B00ESALB0G,0.200465
4,B01FXHSYKM,0.185661
5,B00413QUHE,0.184113
6,B097RLBYQN,0.181542
7,B003KTM9CA,0.177189
8,B087KBMV6D,0.176095
9,B0018OMK82,0.172820



User: AE222Y4WTST6BUZ4J5Y2H6QMBITQ


,parent_asin,score
0,B000ARJGC6,0.240995
1,B0099DPY6E,0.187117
2,B002HJ377A,0.170679
3,B005DJ227C,0.169649
4,B087ZJS6Z5,0.158132
5,B0002TV2RE,0.155938
6,B005GI8UZ8,0.154607
7,B00IEYGUHQ,0.150283
8,B000TA8LJ8,0.147928
9,B007AAYWZC,0.144289



User: AE2237VVHT5JDS6PQAGC4SUJBBXQ


,parent_asin,score
0,B001D4NA5O,0.000074
1,B09TQ58RLF,0.000073
2,B07X7SN16G,0.000066
3,B005FOCNKQ,0.000064
4,B008HDZ4ZS,0.000064
5,B07TVHSDMQ,0.000058
6,B008HF8NJK,0.000058
7,B00BMM04K6,0.000052
8,B08LM29TPB,0.000051
9,B003JY8182,0.000050


## Production Notes

- Switch to parquet by setting `cfg.file_format = "parquet"` and updating file paths.
- Keep sparse dtypes (`float32` values, `int32` indices) for memory efficiency.
- For final offline evaluation on large data, keep warm-start filtering and evaluate per user (not per interaction row).
